# rackmains / ups / battery

The default model of day one for the rack UPS battery: the guard thresholds as constants. The fitted model follows once the `wasm` flavor exists.

| Term | In This Notebook |
|---|---|
| **Intent** | Give the shutdown guard conservative thresholds from its first release, through the same project path every later model uses |
| **Project Flavor** | `static` now, `wasm` for the fitted model (`spec.md` 3.2) |
| **Data Splits** | `golden` the fixed samples and expected outputs; `training`, `validation` and `backtest` follow with the fit |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `library` on the path, load `env`, and declare the model name, version, dataset names and the constants (`spec.md` 6.6).

In [ ]:
from library.data import golden_data
from library.env import init_env
from library.project import consumer_project_dir, provenance

env = init_env()

WORKSPACE = "rackmains/ups/battery"
MODEL_NAME = "rackmains_ups_battery"
MODEL_VERSION = "static-1"
GOLDEN = "ups_battery_golden"
CONSTANTS = {
    "rackmains_guard_shed_max_seconds": 120,
    "rackmains_guard_halt_max_seconds": 450,
    "rackmains_guard_mains_stable_seconds": 300,
    "rackmains_guard_halt_delay_seconds": 120,
}

## Capture Data

- **Stage:** 1, capture data (`spec.md` 7.1).
- **Intent:** write the `golden` rows once: a first call with no state, a row mid-drain, a row with the replace flag raised, and a request with a required input missing, which must give a signature error (`spec.md` 8).

In [ ]:
def golden_rows():
    clock = "rackmains_clock_epoch_seconds"
    util = "rackmains_output_utilisation_pct"
    battery = "rackmains_ups_on_battery"
    replace = "rackmains_battery_replace"
    state = "rackmains_model_state_in"
    return {
        "case": ["first call with no state", "mid-drain with state", "replace flag raised", "required input missing"],
        "expect": ["ok", "ok", "ok", "error"],
        clock: [1793355300.0, 1793355330.0, 1793355360.0, 1793355390.0],
        util: [13, 13, 12, None],
        battery: [1, 1, 0, 1],
        replace: [0, 0, 1, 0],
        state: [None, "0.0 1235.0 0.0 1.0 1.0 1793355329.0", "0.0 0.0 600.0 0.0 1.0 1793355359.0", None],
    }


golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_rows)
golden_pl

## Package

- **Stage:** 5, package.
- **Intent:** build the signature and `golden.json`, write the `static` project into the rackmains consumer as its default and verify it (R1, R3, R8).

In [ ]:
from asystem.model import build_signature, package_project, register_project, signature_input, signature_output, signature_state

signature = build_signature(
    MODEL_NAME,
    [MODEL_VERSION],
    [
        signature_input("rackmains_clock_epoch_seconds", "seconds", "Time of the poll as seconds since the Unix epoch. The ISO text timestamp stays beside it in the record and is not sent."),
        signature_input(
            "rackmains_output_utilisation_pct",
            "percent",
            "Load on the UPS output as a whole percent of its rated capacity. Not watts: whether the rating is volt-amperes or watts is not known.",
            datatype="INT64",
        ),
        signature_input(
            "rackmains_ups_on_battery",
            "flag",
            "1 while the UPS is running from its battery because the mains has failed, 0 on mains. Held at 0 during a deliberate battery calibration. The status text it is derived from stays beside it and is not sent.",
            datatype="INT64",
        ),
        signature_input("rackmains_battery_replace", "flag", "1 while the UPS reports the battery needs replacing.", datatype="INT64"),
        signature_input(
            "rackmains_model_state_in",
            "state",
            "The state the model returned on the previous call, carried unchanged by the caller. Absent on the first call, when the model starts from its initial state with the history unknown.",
            required=False,
            shape=[-1],
        ),
    ],
    [
        signature_output("rackmains_guard_shed_max_seconds", "seconds", "Seconds on battery after which the guard puts the follower servers to sleep, whatever else is estimated."),
        signature_output("rackmains_guard_halt_max_seconds", "seconds", "Seconds on battery after which the guard halts max and has the UPS cut and restore its output. Later than the shed time."),
        signature_output("rackmains_guard_mains_stable_seconds", "seconds", "Seconds of unbroken mains after which the on-battery clock resets and any server put to sleep is woken."),
        signature_output(
            "rackmains_guard_halt_delay_seconds", "seconds", "Seconds the UPS waits before cutting its output after the guard asks it to shut down. Long enough for max to finish going to sleep."
        ),
        signature_output(
            "rackmains_guard_shed_lead_seconds",
            "seconds",
            "Optional. Put the follower servers to sleep when the estimated run time left falls to this: the measured time the shed takes plus a margin.",
            required=False,
        ),
        signature_output(
            "rackmains_guard_halt_lead_seconds", "seconds", "Optional. Halt max when the estimated run time left falls to this: the measured time the halt takes plus a margin.", required=False
        ),
        signature_output(
            "rackmains_battery_remaining_pct",
            "percent",
            "Optional. Estimated usable battery left as a whole percent, 100 when full, rounded down. Informational: the guard does not act on it.",
            required=False,
        ),
        signature_output("rackmains_battery_remaining_seconds", "seconds", "Optional. Estimated run time left at the current load, rounded down, so it never overstates.", required=False),
        signature_output(
            "rackmains_model_energy_used_wh",
            "watt hours",
            "Optional. Energy the model counts as used since the battery was last full, in the current outage and the unrecovered part of earlier ones.",
            datatype="FP64",
            required=False,
        ),
        signature_output(
            "rackmains_model_history_known", "flag", "Optional. 1 once the model knows the battery had recovered before this outage, 0 while it is assuming a partly drained battery.", required=False
        ),
        signature_output(
            "rackmains_model_recalibrate",
            "flag",
            "Optional. 1 when the fit should be redone because it is old, the battery has been replaced or observed drains no longer match its predictions. The reason is in the response parameters.",
            required=False,
        ),
        signature_output(
            "rackmains_model_state_out",
            "state",
            "Optional. The state to carry to the next call, in the order the signature lists. Absent for a model with no state.",
            datatype="FP64",
            required=False,
            shape=[-1],
        ),
    ],
    state=[
        signature_state("unrecovered_pct_seconds", "percent seconds", "Utilisation integrated over the time on battery in earlier outages and not yet recovered."),
        signature_state("outage_pct_seconds", "percent seconds", "Utilisation integrated over the time on battery in the current outage."),
        signature_state("mains_seconds", "seconds", "Seconds on mains since the last outage ended."),
        signature_state("was_on_battery", "flag", "1 if the previous call was on battery, 0 otherwise."),
        signature_state("history_known", "flag", "1 once the battery is known to have recovered, 0 until then."),
        signature_state("last_epoch_seconds", "seconds", "The time of the previous call, 0 before the first."),
    ],
)

project_dir = consumer_project_dir("max/rackmains", "default")
manifest = package_project(
    project_dir,
    MODEL_NAME,
    MODEL_VERSION,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance("manual", "none, the rollout A starting values", [golden_path], "a conservative default, not fitted"),
    constants=CONSTANTS,
    sequence="baseline calls",
)
print(manifest["name"], manifest["model_version"], manifest["flavor"], "written to", project_dir)

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `candidate`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_project(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], aliases=("candidate",))
print("registered version", version, "with alias candidate, set release by hand after review")